In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from shutil import copyfile
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("🚀 Starting Self-Contained End-to-End Research Pipeline...\n")

# ==========================================
# STEP 1: DIRECT KAGGLE DATASET DOWNLOAD
# ==========================================
print("🔑 Kaggle Authentication Setup...")
kaggle_username = input("Enter your Kaggle Username: ")
kaggle_key = input("Enter your Kaggle API Key: ")

# Credentials ko environment mein set karna
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

zip_name = 'skin-cancer-mnist-ham10000.zip'
raw_extract_path = '/content/raw_kaggle_data'

if not os.path.exists(zip_name) and not os.path.exists(raw_extract_path):
    print("\n⬇️ Downloading dataset directly from Kaggle... (Isme 1-2 minute lagenge) ⏳")
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
    print("✅ Download Complete!")

if not os.path.exists(raw_extract_path):
    print("📦 Extracting raw Kaggle files... ⏳")
    import zipfile
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(raw_extract_path)
    print("✅ Raw Extraction Complete!")

# ==========================================
# STEP 2: PATIENT-LEVEL SPLIT (PREVENT LEAKAGE)
# ==========================================
print("\n🛡️ Performing Patient-Level Split based on 'lesion_id' to prevent Data Leakage...")

metadata_path = os.path.join(raw_extract_path, 'HAM10000_metadata.csv')
df = pd.read_csv(metadata_path)

# Unique lesions nikal kar stratify split karna ham10000 ki 7 classes par
unique_lesions = df.groupby('lesion_id').first().reset_index()
train_lesions, val_lesions = train_test_split(
    unique_lesions['lesion_id'],
    test_size=0.2,
    random_state=42,
    stratify=unique_lesions['dx']
)

train_lesions_set = set(train_lesions)
val_lesions_set = set(val_lesions)

# Clean directory structure banana
base_dir = '/content/skin_disease_data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')

runs_dir = '/content/runs/mobilenet_exp'
weights_dir = os.path.join(runs_dir, 'weights')
os.makedirs(weights_dir, exist_ok=True)

# Images source directories mapping
img_dirs = [
    os.path.join(raw_extract_path, 'HAM10000_images_part_1'),
    os.path.join(raw_extract_path, 'HAM10000_images_part_2')
]

# Images ko unki respective class folders mein sort karna
print("Organizing images into Train and Validation directories... ⏳")
for idx, row in df.iterrows():
    img_id = row['image_id']
    lesion_id = row['lesion_id']
    label = row['dx']

    # Target folder faisla karna
    target_sub = train_dir if lesion_id in train_lesions_set else val_dir
    target_class_dir = os.path.join(target_sub, label)
    os.makedirs(target_class_dir, exist_ok=True)

    # Image source dhoond kar copy karna
    img_found = False
    for d in img_dirs:
        src_img_path = os.path.join(d, f"{img_id}.jpg")
        if os.path.exists(src_img_path):
            copyfile(src_img_path, os.path.join(target_class_dir, f"{img_id}.jpg"))
            img_found = True
            break

print("✅ Dataset successfully split and balanced!")

# ==========================================
# STEP 3: DATA GENERATORS & CLASS WEIGHTS
# ==========================================
print("\nConfiguring Data Augmentation and Preprocessing...")
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, class_mode='categorical')
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=False)

class_names = list(train_generator.class_indices.keys())

# Class Imbalance Handle karna with Weights
class_weights_array = compute_class_weight(class_weight='balanced', classes=np.unique(train_generator.classes), y=train_generator.classes)
class_weights = dict(enumerate(class_weights_array))

# ==========================================
# STEP 4: BASELINE TRAINING (5 EPOCHS)
# ==========================================
print("\n🏗️ Loading MobileNetV2 Architecture...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(7, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

baseline_path = os.path.join(weights_dir, 'baseline_model.h5')
callbacks = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
             ModelCheckpoint(baseline_path, save_best_only=True, monitor='val_accuracy')]

print("\n🔥 Training Classification Head (Warm-up Phase)...")
history_base = model.fit(train_generator, epochs=5, validation_data=val_generator, class_weight=class_weights, callbacks=callbacks)

# ==========================================
# STEP 5: FINE-TUNING PHASE (7 EPOCHS)
# ==========================================
print("\n🔄 Unfreezing Top Layers for Deep Fine-Tuning...")
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

best_path = os.path.join(weights_dir, 'best_model.h5')
callbacks_ft = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
                ModelCheckpoint(best_path, save_best_only=True, monitor='val_accuracy')]

print("\n🔥 Fine-Tuning Model...")
history_ft = model.fit(train_generator, epochs=7, validation_data=val_generator, class_weight=class_weights, callbacks=callbacks_ft)

# ==========================================
# STEP 6: POST-TRAINING GENUINE EVALUATION & PLOTS
# ==========================================
print("\n📊 Generating Genuine Evaluation Metrics and Plots for Research Paper...")

# Continuous Performance Curves
acc = history_base.history['accuracy'] + history_ft.history['accuracy']
val_acc = history_base.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history_base.history['loss'] + history_ft.history['loss']
val_loss = history_base.history['val_loss'] + history_ft.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='blue', linewidth=1.5)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='green', linewidth=1.5)
plt.title('Model Accuracy Progress Across Phases')
plt.xlabel('Total Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='red', linewidth=1.5)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange', linewidth=1.5)
plt.title('Model Loss Progress Across Phases')
plt.xlabel('Total Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig(os.path.join(runs_dir, 'metrics_curve.png'), dpi=300)
plt.close()

# Confusion Matrix Generation on Unseen Val Set
print("Generating Confusion Matrix...")
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_generator.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - MobileNetV2 Experimental Pipeline')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.savefig(os.path.join(runs_dir, 'confusion_matrix.png'), dpi=300)
plt.close()

# Classification Report saving as CSV
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report.to_csv(os.path.join(runs_dir, 'classification_report.csv'))

print(f"\n🎉 ALL ASSETS SAVED SUCCESSFULLY IN LOCAL DIRECTORY: {runs_dir}")

# Compressing experiments directory for direct laptop extraction
import shutil
shutil.make_archive('/content/mobilenet_runs_complete', 'zip', '/content/runs/mobilenet_exp')
print("📦 Directory zipped successfully. Downloading to your laptop now...")

🚀 Starting Self-Contained End-to-End Research Pipeline...

🔑 Kaggle Authentication Setup...
Enter your Kaggle Username: haroonbaloshi868
Enter your Kaggle API Key: 203ee5d839ff48123961759616aa31b9

⬇️ Downloading dataset directly from Kaggle... (Isme 1-2 minute lagenge) ⏳
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:41<00:00, 134MB/s]

✅ Download Complete!
📦 Extracting raw Kaggle files... ⏳
✅ Raw Extraction Complete!

🛡️ Performing Patient-Level Split based on 'lesion_id' to prevent Data Leakage...
Organizing images into Train and Validation directories... ⏳
✅ Dataset successfully split and balanced!

Configuring Data Augmentation and Preprocessing...
Found 8017 images belonging to 7 classes.
Found 1998 images belonging to 7 classes.

🏗️ Loading MobileNetV2 Architecture...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

🔥 Training Classification Head (Warm-up Phase)...
Epoch 1/5
251/251 ━━━━━━━━━━━━━━━

251/251 ━━━━━━━━━━━━━━━━━━━━ 206s 743ms/step - accuracy: 0.3503 - loss: 1.7929 - precision: 0.5372 - recall: 0.0983 - val_accuracy: 0.4174 - val_loss: 1.5846 - val_precision: 0.5911 - val_recall: 0.0666
Epoch 2/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 538ms/step - accuracy: 0.4007 - loss: 1.5749 - precision: 0.6300 - recall: 0.1492

251/251 ━━━━━━━━━━━━━━━━━━━━ 149s 594ms/step - accuracy: 0.4310 - loss: 1.5169 - precision: 0.6761 - recall: 0.1854 - val_accuracy: 0.6081 - val_loss: 1.1618 - val_precision: 0.8529 - val_recall: 0.3338
Epoch 3/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 143s 570ms/step - accuracy: 0.4659 - loss: 1.4452 - precision: 0.6907 - recall: 0.2259 - val_accuracy: 0.5761 - val_loss: 1.2707 - val_precision: 0.8636 - val_recall: 0.2883
Epoch 4/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - accuracy: 0.4960 - loss: 1.3725 - precision: 0.7385 - recall: 0.2620

251/251 ━━━━━━━━━━━━━━━━━━━━ 145s 577ms/step - accuracy: 0.4795 - loss: 1.3673 - precision: 0.7096 - recall: 0.2591 - val_accuracy: 0.6667 - val_loss: 1.0313 - val_precision: 0.8992 - val_recall: 0.4064
Epoch 5/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 143s 571ms/step - accuracy: 0.4863 - loss: 1.3282 - precision: 0.7021 - recall: 0.2490 - val_accuracy: 0.5350 - val_loss: 1.2918 - val_precision: 0.7845 - val_recall: 0.3609

🔄 Unfreezing Top Layers for Deep Fine-Tuning...

🔥 Fine-Tuning Model...
Epoch 1/7
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 577ms/step - accuracy: 0.3774 - loss: 1.7269 - precision: 0.6549 - recall: 0.1276

251/251 ━━━━━━━━━━━━━━━━━━━━ 190s 665ms/step - accuracy: 0.3914 - loss: 1.6458 - precision: 0.6846 - recall: 0.1441 - val_accuracy: 0.6767 - val_loss: 0.9598 - val_precision: 0.8644 - val_recall: 0.4660
Epoch 2/7
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 529ms/step - accuracy: 0.4402 - loss: 1.4825 - precision: 0.7147 - recall: 0.1978

251/251 ━━━━━━━━━━━━━━━━━━━━ 147s 583ms/step - accuracy: 0.4498 - loss: 1.4475 - precision: 0.7242 - recall: 0.2087 - val_accuracy: 0.6822 - val_loss: 0.9252 - val_precision: 0.8597 - val_recall: 0.5060
Epoch 3/7
251/251 ━━━━━━━━━━━━━━━━━━━━ 148s 591ms/step - accuracy: 0.4882 - loss: 1.3632 - precision: 0.7518 - recall: 0.2720 - val_accuracy: 0.6767 - val_loss: 0.9231 - val_precision: 0.8426 - val_recall: 0.5090
Epoch 4/7
251/251 ━━━━━━━━━━━━━━━━━━━━ 148s 591ms/step - accuracy: 0.5153 - loss: 1.2628 - precision: 0.7430 - recall: 0.3007 - val_accuracy: 0.6727 - val_loss: 0.9279 - val_precision: 0.8381 - val_recall: 0.5155
Epoch 5/7
251/251 ━━━━━━━━━━━━━━━━━━━━ 150s 599ms/step - accuracy: 0.5357 - loss: 1.2143 - precision: 0.7570 - recall: 0.3416 - val_accuracy: 0.6672 - val_loss: 0.9455 - val_precision: 0.8062 - val_recall: 0.5165
Epoch 6/7
251/251 ━━━━━━━━━━━━━━━━━━━━ 148s 586ms/step - accuracy: 0.5458 - loss: 1.1646 - precision: 0.7416 - recall: 0.3530 - val_accuracy: 0.6637 - val_los